In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, joblib
from pathlib import Path

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from scipy.stats import loguniform, randint

SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

print("Setup OK · Versions :")
import sklearn; print(f"  scikit-learn : {sklearn.__version__}")
print(f"  pandas       : {pd.__version__}")
print(f"  numpy        : {np.__version__}")

Setup OK · Versions :
  scikit-learn : 1.8.0
  pandas       : 3.0.3
  numpy        : 2.4.6


In [2]:
df = pd.read_csv("listings.csv")
print(f"✓ Chargé depuis CSV : {df.shape}")

df.head(3)

✓ Chargé depuis CSV : (84055, 79)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,...,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,3109,https://www.airbnb.com/rooms/3109,20250606142312,2025-06-20,city scrape,zen and calm,Lovely Appartment with one bedroom with a Quee...,Good restaurants<br />very close the Montparna...,https://a0.muscache.com/pictures/miso/Hosting-...,3631,https://www.airbnb.com/users/show/3631,Anne,2008-10-14,"Paris, France",NaN,...,2025-06-03,5.00,5.00,5.00,5.00,5.00,5.00,5.00,7511409139079,f,1,1,0,0,0.08
1,5396,https://www.airbnb.com/rooms/5396,20250606142312,2025-06-19,city scrape,Your perfect Paris studio on Île Saint-Louis,"Cozy, well-appointed and graciously designed s...","You are within walking distance to the Louvre,...",https://a0.muscache.com/pictures/52413/f9bf76f...,7903,https://www.airbnb.com/users/show/7903,Borzou,2009-02-14,"Paris, France",We have spent a lot of time traveling for work...,...,2025-06-05,4.62,4.64,4.59,4.81,4.85,4.95,4.59,7510402838018,f,1,1,0,0,2.32
2,7397,https://www.airbnb.com/rooms/7397,20250606142312,2025-06-20,city scrape,MARAIS - 2ROOMS APT - 2/4 PEOPLE,"VERY CONVENIENT, WITH THE BEST LOCATION !",NaN,https://a0.muscache.com/pictures/67928287/330b...,2626,https://www.airbnb.com/users/show/2626,Franck,2008-08-30,"Paris, France","I am a writer,54, author of novels, books of l...",...,2025-06-03,4.73,4.80,4.45,4.92,4.89,4.94,4.74,7510400829623,f,1,1,0,0,2.20


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 84055 entries, 0 to 84054
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            84055 non-null  int64  
 1   listing_url                                   84055 non-null  str    
 2   scrape_id                                     84055 non-null  int64  
 3   last_scraped                                  84055 non-null  str    
 4   source                                        84055 non-null  str    
 5   name                                          84055 non-null  str    
 6   description                                   81177 non-null  str    
 7   neighborhood_overview                         41178 non-null  str    
 8   picture_url                                   84054 non-null  str    
 9   host_id                                       84055 non-null  int64  
 1

In [4]:
print("Shape :", df.shape)
display(df.describe())

Shape : (84055, 79)


,id,scrape_id,host_id,host_listings_count,host_total_listings_count,neighbourhood_group_cleansed,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,minimum_minimum_nights,...,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,8.405500e+04,8.405500e+04,8.405500e+04,84029.000000,84029.000000,0.0,84055.000000,84055.000000,84055.000000,54297.000000,76918.000000,54007.000000,84055.000000,8.405500e+04,84055.000000,...,84055.000000,84055.000000,5.396300e+04,64498.000000,64486.000000,64489.000000,64479.000000,64488.000000,64480.000000,64479.000000,84055.000000,84055.000000,84055.000000,84055.000000,64498.000000
mean,6.347984e+17,2.025061e+13,1.857889e+08,31.938843,41.159861,NaN,48.864047,2.342917,3.225198,1.211503,1.343457,1.830726,43.545512,5.902691e+02,42.508619,...,5.692975,61.367700,1.817894e+04,4.732790,4.776988,4.667760,4.814846,4.833622,4.823588,4.631523,25.369758,24.193016,1.044031,0.014586,1.045166
std,5.402776e+17,0.000000e+00,2.076564e+08,120.838795,157.249944,NaN,0.018094,0.034203,1.684463,0.555283,0.909540,1.190300,109.597333,3.449318e+04,109.034033,...,11.875125,85.354511,6.508574e+04,0.389992,0.368886,0.438217,0.354538,0.354508,0.303654,0.430997,97.944143,96.282841,10.727246,0.259222,1.350142
min,3.109000e+03,2.025061e+13,2.750000e+02,0.000000,0.000000,NaN,48.815890,2.229896,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000e+00,1.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.010000
25%,3.150505e+07,2.025061e+13,1.948488e+07,1.000000,1.000000,NaN,48.850758,2.320790,2.000000,1.000000,1.000000,1.000000,2.000000,4.500000e+01,1.000000,...,0.000000,0.000000,0.000000e+00,4.640000,4.710000,4.540000,4.780000,4.810000,4.770000,4.500000,1.000000,1.000000,0.000000,0.000000,0.180000
50%,8.318022e+17,2.025061e+13,7.315135e+07,1.000000,2.000000,NaN,48.865321,2.346677,3.000000,1.000000,1.000000,2.000000,3.000000,3.650000e+02,3.000000,...,0.000000,10.000000,6.240000e+03,4.840000,4.880000,4.800000,4.920000,4.950000,4.920000,4.730000,1.000000,1.000000,0.000000,0.000000,0.560000
75%,1.118822e+18,2.025061e+13,3.389546e+08,4.000000,6.000000,NaN,48.878630,2.367945,4.000000,1.000000,2.000000,2.000000,6.000000,1.125000e+03,6.000000,...,6.000000,104.000000,2.079650e+04,5.000000,5.000000,4.980000,5.000000,5.000000,5.000000,4.890000,3.000000,2.000000,0.000000,0.000000,1.400000
max,1.437099e+18,2.025061e+13,7.007037e+08,7954.000000,8721.000000,NaN,48.901670,2.468360,16.000000,42.000000,41.000000,17.000000,1000.000000,1.000000e+07,1000.000000,...,786.000000,255.000000,3.655680e+06,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,835.000000,835.000000,166.000000,9.000000,67.160000


In [5]:
print(df.isnull().sum())

id                                                  0
listing_url                                         0
scrape_id                                           0
last_scraped                                        0
source                                              0
                                                ...  
calculated_host_listings_count                      0
calculated_host_listings_count_entire_homes         0
calculated_host_listings_count_private_rooms        0
calculated_host_listings_count_shared_rooms         0
reviews_per_month                               19557
Length: 79, dtype: int64


In [6]:
cols_bool = ['host_has_profile_pic', 'host_identity_verified']
for col in cols_bool:
    # Le map transforme 't' en 1 et 'f' en 0. Les valeurs manquantes restent NaN pour le moment.
    df[col] = df[col].map({'t': 1, 'f': 0})
valeurs_uniques = df.nunique().sort_values()[30:]
print(valeurs_uniques)

calculated_host_listings_count_entire_homes       94
estimated_occupancy_l365d                         98
host_acceptance_rate                             101
minimum_nights                                   105
minimum_minimum_nights                           118
maximum_minimum_nights                           137
number_of_reviews_ly                             140
number_of_reviews_ltm                            142
host_listings_count                              147
review_scores_location                           166
review_scores_communication                      168
review_scores_accuracy                           170
review_scores_checkin                            173
review_scores_rating                             181
review_scores_value                              184
host_total_listings_count                        192
availability_eoy                                 199
review_scores_cleanliness                        203
maximum_nights                                

In [7]:
df['price'] = df['price'].str.replace(r'[\$,]', '', regex=True).astype(float)
df['host_acceptance_rate'] = df['host_acceptance_rate'].str.replace('%', '', regex=False).astype('Int64')
df['host_response_rate'] = df['host_response_rate'].str.replace('%', '', regex=False).astype('Int64')
corrs = df.select_dtypes(include=np.number).corr()['price'].sort_values(ascending=False)
top_corr = corrs.head(40)
print(top_corr)

price                                           1.000000
estimated_revenue_l365d                         0.464334
bathrooms                                       0.234650
accommodates                                    0.233318
bedrooms                                        0.205691
beds                                            0.181490
availability_30                                 0.117645
availability_eoy                                0.102265
availability_365                                0.099006
availability_60                                 0.096502
availability_90                                 0.096237
maximum_nights                                  0.074676
host_listings_count                             0.068375
host_total_listings_count                       0.067636
host_acceptance_rate                            0.047423
host_id                                         0.040676
id                                              0.036011
review_scores_cleanliness      

In [8]:
# 1. Suppression des colonnes 100% vides, des textes longs et des risques de fuite
cols_to_drop = [
    'id', 'listing_url', 'scrape_id', 'last_scraped','source', 'name', 'picture_url', 'host_id', 'host_url', 'host_name', 'neighbourhood_group_cleansed', 'bathrooms', 'beds', 'calendar_updated', 
    'description', 'neighborhood_overview', 'neighbourhood', 'host_about', 'host_thumbnail_url', 'host_picture_url', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm',
    'calendar_last_scraped', 'has_availability', 'reviews_per_month', 'license', 'host_neighbourhood', 'host_location','host_since', 'first_review', 'last_review'
]
# On utilise errors='ignore' au cas où certaines auraient déjà été supprimées
df_airbnb = df.drop(columns=cols_to_drop, errors='ignore')

# 2. Parsing de bathrooms_text
# On utilise une expression régulière pour extraire le premier nombre (entier ou décimal)
df_airbnb['bathrooms'] = df_airbnb['bathrooms_text'].str.extract(r'(\d+\.?\d*)').astype(float)
# On supprime la colonne texte d'origine pour ne garder que la feature numérique
df_airbnb = df_airbnb.drop(columns=['bathrooms_text'])

# 3. Parsing des amenities (Extraction des 5 features minimum exigées)
# On définit un dictionnaire avec le nom de la future colonne et le mot-clé à chercher
amenities_to_extract = {
    'has_wifi': 'Wifi',
    'has_tv': 'TV',
    'has_kitchen': 'Kitchen',
    'has_ac': 'Air conditioning',
    'has_elevator': 'Elevator'
}

for col_name, keyword in amenities_to_extract.items():
    # str.contains cherche le mot-clé. na=False permet de mettre 0 si la ligne est NaN.
    # astype(int) transforme le True/False en 1/0
    df_airbnb[col_name] = df_airbnb['amenities'].str.contains(keyword, case=False, na=False).astype(int)

# On supprime la colonne JSON brute qui n'est plus utile pour le modèle
df_airbnb = df_airbnb.drop(columns=['amenities'])
print(df_airbnb.shape)
df_airbnb.head(5)


(84055, 52)


,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,bathrooms,has_wifi,has_tv,has_kitchen,has_ac,has_elevator
0,within an hour,100,100,f,1.0,1.0,"['email', 'phone']",1.0,1.0,Observatoire,48.83191,2.31870,Entire rental unit,Entire home/apt,2,...,5.00,5.00,5.00,5.00,f,1,1,0,0,1.0,1,0,1,0,0
1,within an hour,100,95,f,2.0,4.0,"['email', 'phone']",1.0,1.0,Hôtel-de-Ville,48.85247,2.35835,Entire rental unit,Entire home/apt,2,...,4.81,4.85,4.95,4.59,f,1,1,0,0,1.0,1,1,1,0,0
2,within an hour,100,67,t,1.0,10.0,"['email', 'phone']",1.0,1.0,Hôtel-de-Ville,48.85909,2.35315,Entire rental unit,Entire home/apt,4,...,4.92,4.89,4.94,4.74,f,1,1,0,0,1.0,1,1,1,0,0
3,NaN,<NA>,<NA>,f,1.0,1.0,['phone'],1.0,1.0,Opéra,48.87417,2.34245,Entire rental unit,Entire home/apt,3,...,5.00,5.00,5.00,5.00,f,1,1,0,0,1.0,1,1,1,0,0
4,NaN,<NA>,<NA>,f,2.0,4.0,"['email', 'phone']",1.0,1.0,Louvre,48.86006,2.34863,Entire rental unit,Entire home/apt,1,...,NaN,NaN,NaN,NaN,f,1,1,0,0,1.0,1,0,1,0,1


In [9]:
df_airbnb.info()

<class 'pandas.DataFrame'>
RangeIndex: 84055 entries, 0 to 84054
Data columns (total 52 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   host_response_time                            52663 non-null  str    
 1   host_response_rate                            52663 non-null  Int64  
 2   host_acceptance_rate                          57913 non-null  Int64  
 3   host_is_superhost                             82227 non-null  str    
 4   host_listings_count                           84029 non-null  float64
 5   host_total_listings_count                     84029 non-null  float64
 6   host_verifications                            84029 non-null  str    
 7   host_has_profile_pic                          84029 non-null  float64
 8   host_identity_verified                        84029 non-null  float64
 9   neighbourhood_cleansed                        84055 non-null  str    
 1

In [10]:
corrs = df_airbnb.select_dtypes(include=np.number).corr()['price'].sort_values(ascending=False)
top_corr = corrs
# print(top_corr)
display(top_corr)

price                                           1.000000
estimated_revenue_l365d                         0.464334
bathrooms                                       0.236619
accommodates                                    0.233318
bedrooms                                        0.205691
has_ac                                          0.134455
availability_30                                 0.117645
availability_eoy                                0.102265
availability_365                                0.099006
availability_60                                 0.096502
availability_90                                 0.096237
has_tv                                          0.086983
maximum_nights                                  0.074676
host_listings_count                             0.068375
host_total_listings_count                       0.067636
host_acceptance_rate                            0.047423
has_elevator                                    0.039241
review_scores_cleanliness      

In [11]:
df_airbnb.isnull().sum()

host_response_time                              31392
host_response_rate                              31392
host_acceptance_rate                            26142
host_is_superhost                                1828
host_listings_count                                26
host_total_listings_count                          26
host_verifications                                 26
host_has_profile_pic                               26
host_identity_verified                             26
neighbourhood_cleansed                              0
latitude                                            0
longitude                                           0
property_type                                       0
room_type                                           0
accommodates                                        0
bedrooms                                         7137
price                                           30092
minimum_nights                                      0
maximum_nights              

In [12]:
df_airbnb['price'].isnull().sum()

np.int64(30092)

In [13]:
df_airbnb['host_acceptance_rate'].head(20)

0      100
1       95
2       67
3     <NA>
4     <NA>
5       34
6       32
7     <NA>
8      100
9     <NA>
10      40
11      40
12      40
13      40
14      57
15      86
16     100
17     100
18      83
19      73
Name: host_acceptance_rate, dtype: Int64

In [14]:
missing = (
    df_airbnb.isnull()
    .sum()
    .reset_index()
    .rename(columns={'index': 'feature', 0: 'missing_count'})
)

df_airbnb.head

# print(missing)

missing['column_type'] = missing['feature'].map(df_airbnb.dtypes.astype(str))
missing['na_percentage'] = missing['missing_count'] / len(df_airbnb) * 100
missing['price_correlation'] = missing['feature'].map(corrs)

# display(
#     missing.loc[
#         (missing['price_correlation'] > 0.05) | (missing['price_correlation'] < -0.05)
#     ].sort_values('price_correlation', ascending=False)
# )

# print("Nb de features numériques avec corr >= 0.05 & < -0.05 : " + str(len(missing[missing['price_correlation'] > 0.05])))

In [15]:
display(
    missing.loc[
        (missing['price_correlation'] > 0.05) & (missing['missing_count'] > 0)
    ].sort_values('price_correlation', ascending=False)
)

display(
    missing.loc[
        (missing['price_correlation'] < -0.05) & (missing['missing_count'] > 0)
    ]
)

,feature,missing_count,column_type,na_percentage,price_correlation
16,price,30092,float64,35.800369,1.000000
33,estimated_revenue_l365d,30092,float64,35.800369,0.464334
46,bathrooms,545,float64,0.648385,0.236619
15,bedrooms,7137,float64,8.490869,0.205691
4,host_listings_count,26,float64,0.030932,0.068375
5,host_total_listings_count,26,float64,0.030932,0.067636


,feature,missing_count,column_type,na_percentage,price_correlation


In [16]:
print(missing.shape)
display(missing.head(20))

(52, 5)


,feature,missing_count,column_type,na_percentage,price_correlation
0,host_response_time,31392,str,37.346975,NaN
1,host_response_rate,31392,Int64,37.346975,0.008692
2,host_acceptance_rate,26142,Int64,31.101065,0.047423
3,host_is_superhost,1828,str,2.174767,NaN
4,host_listings_count,26,float64,0.030932,0.068375
5,host_total_listings_count,26,float64,0.030932,0.067636
6,host_verifications,26,str,0.030932,NaN
7,host_has_profile_pic,26,float64,0.030932,-0.000433
8,host_identity_verified,26,float64,0.030932,0.004752
9,neighbourhood_cleansed,0,str,0.000000,NaN


On garde toutes les features > et/ou < 0.4

## NA Strategy

|NA row|strategy|
|-----|-----|
|price|delete row|
|estimated_revenue_l365d|delete row|
|bathrooms|remplace par 0|
|bedrooms|remplace par 1|
|host_listings_count|delete row|
|host_total_listings_count|delete row|

In [17]:
display(df_airbnb)

,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,bathrooms,has_wifi,has_tv,has_kitchen,has_ac,has_elevator
0,within an hour,100,100,f,1.0,1.0,"['email', 'phone']",1.0,1.0,Observatoire,48.831910,2.318700,Entire rental unit,Entire home/apt,2,...,5.00,5.00,5.00,5.00,f,1,1,0,0,1.0,1,0,1,0,0
1,within an hour,100,95,f,2.0,4.0,"['email', 'phone']",1.0,1.0,Hôtel-de-Ville,48.852470,2.358350,Entire rental unit,Entire home/apt,2,...,4.81,4.85,4.95,4.59,f,1,1,0,0,1.0,1,1,1,0,0
2,within an hour,100,67,t,1.0,10.0,"['email', 'phone']",1.0,1.0,Hôtel-de-Ville,48.859090,2.353150,Entire rental unit,Entire home/apt,4,...,4.92,4.89,4.94,4.74,f,1,1,0,0,1.0,1,1,1,0,0
3,NaN,<NA>,<NA>,f,1.0,1.0,['phone'],1.0,1.0,Opéra,48.874170,2.342450,Entire rental unit,Entire home/apt,3,...,5.00,5.00,5.00,5.00,f,1,1,0,0,1.0,1,1,1,0,0
4,NaN,<NA>,<NA>,f,2.0,4.0,"['email', 'phone']",1.0,1.0,Louvre,48.860060,2.348630,Entire rental unit,Entire home/apt,1,...,NaN,NaN,NaN,NaN,f,1,1,0,0,1.0,1,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84050,within an hour,98,97,f,9.0,15.0,"['email', 'phone']",1.0,1.0,Vaugirard,48.851460,2.291000,Entire rental unit,Entire home/apt,1,...,NaN,NaN,NaN,NaN,t,6,6,0,0,1.0,1,0,1,0,0
84051,NaN,<NA>,100,f,1.0,1.0,"['email', 'phone']",0.0,1.0,Popincourt,48.860340,2.374200,Entire rental unit,Entire home/apt,2,...,NaN,NaN,NaN,NaN,t,1,1,0,0,1.0,0,0,1,0,0
84052,within a few hours,100,0,f,1.0,1.0,"['email', 'phone']",1.0,1.0,Buttes-Montmartre,48.889510,2.334080,Entire rental unit,Entire home/apt,4,...,NaN,NaN,NaN,NaN,f,1,1,0,0,1.0,1,1,1,0,0
84053,NaN,<NA>,<NA>,f,1.0,2.0,"['email', 'phone']",1.0,0.0,Ménilmontant,48.871542,2.381301,Entire rental unit,Entire home/apt,3,...,NaN,NaN,NaN,NaN,t,1,1,0,0,1.0,1,0,1,0,0


In [18]:
selected_cols_corr = missing.loc[missing['price_correlation'].abs() > 0.05, 'feature'].dropna().tolist()
# print(len(selected_cols))
selected_cols_corr.append('latitude')
feat_to_add = [
   'room_type',
   'neighbourhood_cleansed',
   'property_type',
  #  'host_response_time',
]
for f in feat_to_add :
    selected_cols_corr.append(f)
print(selected_cols_corr)

col_to_r = []

count = 0

for col in df_airbnb.columns :
    col_to_r.append(col)

for scol in selected_cols_corr :
    col_to_r.remove(scol)
    print("removed ", scol)
       
df_airbnb = df_airbnb.drop(columns=col_to_r)

print(df_airbnb.columns)

# NA Strategies
df_airbnb = df_airbnb.dropna(subset=['price', 'estimated_revenue_l365d', 'host_listings_count', 'host_total_listings_count'])
df_airbnb.isnull().sum().sort_values(ascending=False)
df_airbnb['bathrooms'] = df_airbnb['bathrooms'].fillna(0)
df_airbnb['bedrooms'] = df_airbnb['bedrooms'].fillna(1)

['host_listings_count', 'host_total_listings_count', 'longitude', 'accommodates', 'bedrooms', 'price', 'maximum_nights', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'bathrooms', 'has_tv', 'has_kitchen', 'has_ac', 'latitude', 'room_type', 'neighbourhood_cleansed', 'property_type']
removed  host_listings_count
removed  host_total_listings_count
removed  longitude
removed  accommodates
removed  bedrooms
removed  price
removed  maximum_nights
removed  availability_30
removed  availability_60
removed  availability_90
removed  availability_365
removed  availability_eoy
removed  number_of_reviews_ly
removed  estimated_occupancy_l365d
removed  estimated_revenue_l365d
removed  bathrooms
removed  has_tv
removed  has_kitchen
removed  has_ac
removed  latitude
removed  room_type
removed  neighbourhood_cleansed
removed  property_type
Index(['host_listings_count', 'host

In [19]:
print("After cleaner : ", len(df_airbnb.columns))
print(df_airbnb.columns)
print(df_airbnb.shape)

display(df_airbnb)

After cleaner :  23
Index(['host_listings_count', 'host_total_listings_count', 'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bedrooms', 'price', 'maximum_nights',
       'availability_30', 'availability_60', 'availability_90', 'availability_365', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'bathrooms',
       'has_tv', 'has_kitchen', 'has_ac'],
      dtype='str')
(53953, 23)


,host_listings_count,host_total_listings_count,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,price,maximum_nights,availability_30,availability_60,availability_90,availability_365,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,bathrooms,has_tv,has_kitchen,has_ac
0,1.0,1.0,Observatoire,48.831910,2.318700,Entire rental unit,Entire home/apt,2,1.0,135.0,30,20,50,80,355,185,0,31,4185.0,1.0,0,1,0
1,2.0,4.0,Hôtel-de-Ville,48.852470,2.358350,Entire rental unit,Entire home/apt,2,0.0,114.0,730,7,37,54,69,69,52,255,29070.0,1.0,1,1,0
2,1.0,10.0,Hôtel-de-Ville,48.859090,2.353150,Entire rental unit,Entire home/apt,4,2.0,149.0,130,0,12,27,197,122,24,255,37995.0,1.0,1,1,0
4,2.0,4.0,Louvre,48.860060,2.348630,Entire rental unit,Entire home/apt,1,1.0,75.0,360,23,53,83,358,190,0,0,0.0,1.0,0,1,0
5,1.0,3.0,Buttes-Montmartre,48.889460,2.358670,Private room in rental unit,Private room,1,1.0,50.0,1125,21,51,81,82,82,2,31,1550.0,1.0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84050,9.0,15.0,Vaugirard,48.851460,2.291000,Entire rental unit,Entire home/apt,1,1.0,78.0,365,5,5,5,5,5,0,0,0.0,1.0,0,1,0
84051,1.0,1.0,Popincourt,48.860340,2.374200,Entire rental unit,Entire home/apt,2,1.0,89.0,365,17,47,77,352,184,0,0,0.0,1.0,0,1,0
84052,1.0,1.0,Buttes-Montmartre,48.889510,2.334080,Entire rental unit,Entire home/apt,4,1.0,105.0,365,10,28,58,333,165,0,0,0.0,1.0,1,1,0
84053,1.0,2.0,Ménilmontant,48.871542,2.381301,Entire rental unit,Entire home/apt,3,1.0,100.0,15,16,31,31,195,123,0,0,0.0,1.0,0,1,0


In [20]:
print(df_airbnb.shape)
display(df_airbnb.describe())
display(df_airbnb.info())

(53953, 23)


,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bedrooms,price,maximum_nights,availability_30,availability_60,availability_90,availability_365,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,bathrooms,has_tv,has_kitchen,has_ac
count,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,53953.000000,5.395300e+04,53953.000000,53953.000000,53953.000000,53953.000000
mean,43.684540,55.818268,48.863842,2.341131,3.380943,1.325413,284.929828,402.167961,9.636165,26.905992,44.419106,188.722203,104.644283,7.363483,80.502789,1.818082e+04,1.208737,0.738717,0.945175,0.177821
std,141.733065,184.618221,0.017628,0.034147,1.787886,0.932594,686.421679,397.140046,9.625280,19.612662,29.554871,119.580880,66.981714,13.547818,90.921993,6.509133e+04,0.562128,0.439338,0.227641,0.382366
min,1.000000,1.000000,48.815890,2.229896,1.000000,0.000000,8.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,48.851110,2.318840,2.000000,1.000000,104.000000,40.000000,1.000000,9.000000,18.000000,73.000000,40.000000,0.000000,0.000000,0.000000e+00,1.000000,0.000000,1.000000,0.000000
50%,1.000000,2.000000,48.865070,2.345401,3.000000,1.000000,161.000000,365.000000,6.000000,25.000000,44.000000,195.000000,111.000000,2.000000,42.000000,6.240000e+03,1.000000,1.000000,1.000000,0.000000
75%,10.000000,14.000000,48.877649,2.364830,4.000000,2.000000,277.000000,365.000000,17.000000,45.000000,72.000000,294.000000,169.000000,9.000000,135.000000,2.080000e+04,1.000000,1.000000,1.000000,0.000000
max,7954.000000,8721.000000,48.901590,2.468360,16.000000,33.000000,30814.000000,1126.000000,30.000000,60.000000,90.000000,365.000000,198.000000,786.000000,255.000000,3.655680e+06,42.000000,1.000000,1.000000,1.000000


<class 'pandas.DataFrame'>
Index: 53953 entries, 0 to 84054
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   host_listings_count        53953 non-null  float64
 1   host_total_listings_count  53953 non-null  float64
 2   neighbourhood_cleansed     53953 non-null  str    
 3   latitude                   53953 non-null  float64
 4   longitude                  53953 non-null  float64
 5   property_type              53953 non-null  str    
 6   room_type                  53953 non-null  str    
 7   accommodates               53953 non-null  int64  
 8   bedrooms                   53953 non-null  float64
 9   price                      53953 non-null  float64
 10  maximum_nights             53953 non-null  int64  
 11  availability_30            53953 non-null  int64  
 12  availability_60            53953 non-null  int64  
 13  availability_90            53953 non-null  int64  
 14  availa

None

# Split des données

In [21]:
cols_features = [col for col in df_airbnb.columns if col != 'price']
X = df_airbnb[cols_features]
y = df_airbnb['price']

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X.shape, y.shape

((53953, 22), (53953,))

# Transformation des données

In [25]:
# num_cols = [*cols_features, 'price'] 
cat_cols = [
   'room_type',
   'neighbourhood_cleansed',
   'property_type',
  #  'host_response_time',
]
num_cols = [c for c in cols_features if c not in cat_cols]

num_pipeline = Pipeline([
    # ('imputer', SimpleImputer(strategy='median')),
  ('scaler', StandardScaler()),
])
cat_pipeline = Pipeline([
  # ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
  ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# preprocessor = ColumnTransformer([
#   ('num', num_pipeline, num_cols),
#   ('cat', cat_pipeline, cat_cols)
# ])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
])

pipe_lr.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse ma